# TAPS-MCC — Trust-Aware Offloading Simulation
Discrete-event simulation of 20 edge servers serving 100–1,000 mobile users, with rogue and compromised (on-off, partly stealthy) edge servers.
It compares **Latency-only**, **Auth-only**, **Auth+Trust** and full **TAPS-MCC** (auth + trust + IDS + privacy), using the crypto timings measured on the Motorola Edge 60 Pro (user) and the Ryzen 5 5600H laptop (edge server).

**Run:** `Runtime → Run all` (about 5 minutes). `TAPS_SIM_results.zip` downloads at the end — upload it to Claude.

## 1. Settings

In [ ]:
SEEDS     = 10                          # repetitions per configuration
USERS     = list(range(100, 1001, 100))   # experiment A
MAL       = [0.0, 0.1, 0.2, 0.3, 0.4]     # experiment B
IDS_TPR   = 0.894   # LightGBM binary detection rate (IDS v2)
IDS_FPR   = 0.0001  # LightGBM binary false-positive rate (IDS v2)

## 2. Simulation model

In [ ]:
# ---------------- TAPS-MCC offloading simulation: core ----------------
import numpy as np, pandas as pd

P = dict(
    N_ES=20, ES_MIPS=150_000,            # 20 edge servers, aggregate capacity per server (MIPS)
    SIM_T=120.0, RATE=0.2,              # simulated seconds; tasks per user per second (Poisson)
    WLAN_MBPS=20.0, WLAN_MS=5.0,        # user<->local ES access link
    MAN_MBPS=1000.0, MAN_MS=5.0,        # ES<->ES metro link (when offloaded to a non-local ES)
    N_REGIONS=4,                        # regional orchestrators (source of indirect trust)
    THETA=0.6, ALPHA=0.4, BETA=0.3, GAMMA=0.3, LAM=0.9, TS0=0.7,   # Eq. 2-3 parameters
    P_ACTIVE=0.7,                       # on-off insider: probability it misbehaves on a task
    P_STEALTH=0.5,                      # share of insider attacks that are stealthy (result looks correct)
    ROGUE_LURE=1.5,                     # rogue ES advertises 1.5x its real capacity to attract tasks
    SESSION_S=60.0,                     # re-authentication interval per user
    IDS_TPR=0.99, IDS_FPR=0.001,        # replace with IDS v2 results
    # measured crypto costs (ms): phone = MU, laptop = ES
    AUTH_MU_MS=0.333, AUTH_ES_MS=0.172, HMAC_MU_MS=0.0041, HMAC_ES_MS=0.0022, PAILLIER_ENC_MS=171.59,
    # device power model (W)
    P_TX=1.3, P_RX=1.0, P_IDLE=0.1, P_CPU=2.0,
)
# name, probability, upload KB, download KB, length MI, deadline s, carries sensitive data
APPS = [("AR", 0.3, 1500, 25, 9000, 2.0, False),
        ("Health", 0.3, 20, 1250, 3000, 2.0, True),
        ("Infotainment", 0.4, 25, 1000, 15000, 3.0, False)]
STRATS = ["Latency-only", "Auth-only", "Auth+Trust", "TAPS-MCC"]


def run(strategy, n_users, mal_ratio, seed, p=P, trace=False):
    rng = np.random.default_rng(seed)
    N, R = p["N_ES"], p["N_REGIONS"]
    n_mal = int(round(mal_ratio * N)); n_rogue = n_mal // 2; n_ins = n_mal - n_rogue
    typ = np.zeros(N, int)                                   # 0 honest, 1 compromised insider, 2 rogue
    perm = rng.permutation(N); typ[perm[:n_ins]] = 1; typ[perm[n_ins:n_ins + n_rogue]] = 2
    mips = np.full(N, float(p["ES_MIPS"])); adv = mips.copy(); adv[typ == 2] *= p["ROGUE_LURE"]
    next_free = np.zeros(N); busy = np.zeros(N)

    use_auth = strategy != "Latency-only"
    use_trust = strategy in ("Auth+Trust", "TAPS-MCC")
    use_ids = use_priv = strategy == "TAPS-MCC"
    if use_ids: a, b, g = p["ALPHA"], p["BETA"], p["GAMMA"]
    else:
        s = p["ALPHA"] + p["BETA"]; a, b, g = p["ALPHA"] / s, p["BETA"] / s, 0.0
    allowed = (typ != 2) if use_auth else np.ones(N, bool)
    succ = np.zeros((R, N)); fail = np.zeros((R, N)); TS = np.full((R, N), p["TS0"])
    flag = np.zeros(N)

    nt = rng.poisson(p["RATE"] * p["SIM_T"] * n_users)
    t_arr = np.sort(rng.uniform(0, p["SIM_T"], nt))
    users = rng.integers(0, n_users, nt)
    probs = np.array([x[1] for x in APPS]); apps = rng.choice(len(APPS), nt, p=probs / probs.sum())
    last_auth = np.full(n_users, -1e18)
    idx = np.arange(N)
    lat = np.empty(nt); done = np.zeros(nt, bool); brch = np.zeros(nt, bool); failed = np.zeros(nt, bool); malo = np.zeros(nt, bool); energy = np.empty(nt)
    tr = []
    if trace:
        h_es = int(np.where(typ == 0)[0][0]); m_es = int(np.where(typ == 1)[0][0]) if n_ins else None

    for i in range(nt):
        u, t = users[i], t_arr[i]
        _, _, up, down, mi, dl, sens = APPS[apps[i]]
        delay = 0.0; cpu_ms = 0.0
        if use_auth:
            if t - last_auth[u] > p["SESSION_S"]:
                delay += (3 * p["WLAN_MS"] + p["AUTH_MU_MS"] + p["AUTH_ES_MS"]) / 1000
                cpu_ms += p["AUTH_MU_MS"]; last_auth[u] = t
            delay += (p["HMAC_MU_MS"] + p["HMAC_ES_MS"]) / 1000; cpu_ms += p["HMAC_MU_MS"]
        if use_priv and sens:
            delay += p["PAILLIER_ENC_MS"] / 1000; cpu_ms += p["PAILLIER_ENC_MS"]
        up_s = up * 8 / 1000 / p["WLAN_MBPS"] + p["WLAN_MS"] / 1000
        arrive = t + delay + up_s
        local = u % N
        man_up = np.where(idx == local, 0.0, p["MAN_MS"] / 1000 + up * 8 / 1000 / p["MAN_MBPS"])
        exp_fin = np.maximum(arrive + man_up, next_free) + mi / adv

        cand = allowed
        r = u % R
        if use_trust:
            ok = allowed & (TS[r] >= p["THETA"])
            cand = ok if ok.any() else allowed & (TS[r] >= TS[r][allowed].max())
        j = int(np.argmin(np.where(cand, exp_fin, np.inf)))

        start = max(arrive + man_up[j], next_free[j]); ex = mi / mips[j]
        next_free[j] = start + ex; busy[j] += ex
        man_dn = 0.0 if j == local else p["MAN_MS"] / 1000 + down * 8 / 1000 / p["MAN_MBPS"]
        down_s = down * 8 / 1000 / p["WLAN_MBPS"] + p["WLAN_MS"] / 1000 + man_dn
        L = start + ex + down_s - t

        active = False; sec_ok = True; breach = False
        if typ[j] == 2: sec_ok = False; breach = True                 # rogue node drops / steals
        elif typ[j] == 1 and rng.random() < p["P_ACTIVE"]:
            active = True; breach = True
            if rng.random() >= p["P_STEALTH"]: sec_ok = False         # visible: drop or tamper (HMAC fails)
        lat[i] = L; malo[i] = typ[j] > 0; brch[i] = breach; done[i] = sec_ok
        failed[i] = (not sec_ok) or L > dl
        energy[i] = p["P_TX"] * up_s + p["P_RX"] * down_s + p["P_IDLE"] * max(0.0, L - up_s - down_s) + p["P_CPU"] * cpu_ms / 1000

        if use_trust:
            if sec_ok: succ[r, j] += 1
            else: fail[r, j] += 1
            DT = (succ[r, j] + 1) / (succ[r, j] + fail[r, j] + 2)
            o = [q for q in range(R) if q != r]
            IT = float(np.mean((succ[o, j] + 1) / (succ[o, j] + fail[o, j] + 2)))
            if use_ids:
                f = rng.random() < (p["IDS_TPR"] if active else p["IDS_FPR"])
                flag[j] = 0.9 * flag[j] + 0.1 * f
            BT = 1 - flag[j]
            TS[r, j] = p["LAM"] * TS[r, j] + (1 - p["LAM"]) * (a * DT + b * IT + g * BT)
            if trace and r == 0 and j in (h_es, m_es):
                tr.append((t, "honest" if j == h_es else "malicious", TS[0, j]))

    out = dict(strategy=strategy, users=n_users, mal_ratio=mal_ratio, seed=seed, tasks=nt,
               latency_ms=float(lat[done].mean() * 1000) if done.any() else np.nan, failure_rate=float(failed.mean()),
               breach_rate=float(brch.mean()),
               malicious_offload_rate=float(malo.mean()), energy_J=float(energy.mean()),
               es_utilization=float(busy.sum() / (N * p["SIM_T"])))
    return (out, pd.DataFrame(tr, columns=["time_s", "es", "trust"])) if trace else out


In [ ]:
P['IDS_TPR'], P['IDS_FPR'] = IDS_TPR, IDS_FPR
import os, time, json, zipfile, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
os.makedirs('sim', exist_ok=True)
def ci(x): return 1.96 * np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0
METRICS = ['latency_ms', 'failure_rate', 'breach_rate', 'malicious_offload_rate', 'energy_J', 'es_utilization']
def summarise(df, by):
    g = df.groupby(by, sort=False)[METRICS]
    out = g.mean().add_suffix('_mean').join(g.agg(ci).add_suffix('_ci95'))
    return out.reset_index()

## 3. Experiment A — scalability (20% malicious edge servers)

In [ ]:
t0 = time.time()
A = pd.DataFrame([run(s, u, 0.2, sd) for s in STRATS for u in USERS for sd in range(SEEDS)])
A.to_csv('sim/A_raw.csv', index=False); SA = summarise(A, ['strategy', 'users']); SA.to_csv('sim/A_summary.csv', index=False)
print(f'{time.time()-t0:.0f}s'); SA.round(4)

## 4. Experiment B — robustness (500 users, 0–40% malicious)

In [ ]:
t0 = time.time()
B = pd.DataFrame([run(s, 500, m, sd) for s in STRATS for m in MAL for sd in range(SEEDS)])
B.to_csv('sim/B_raw.csv', index=False); SB = summarise(B, ['strategy', 'mal_ratio']); SB.to_csv('sim/B_summary.csv', index=False)
print(f'{time.time()-t0:.0f}s'); SB.round(4)

## 5. Experiment C — trust evolution of an honest vs a compromised server

In [ ]:
TR = []
for s in ['Auth+Trust', 'TAPS-MCC']:
    _, tr = run(s, 500, 0.2, 0, trace=True); tr['strategy'] = s; TR.append(tr)
TR = pd.concat(TR); TR.to_csv('sim/C_trust_trace.csv', index=False)

## 6. Experiment D — sensitivity to IDS detection rate (500 users, 20% malicious)

In [ ]:
D = []
for tpr in [0.80, 0.90, 0.95, 0.99]:
    q = dict(P, IDS_TPR=tpr)
    D += [dict(run('TAPS-MCC', 500, 0.2, sd, p=q), ids_tpr=tpr) for sd in range(SEEDS)]
D = pd.DataFrame(D); SD = summarise(D, ['ids_tpr']); SD.to_csv('sim/D_ids_sensitivity.csv', index=False); SD.round(4)

## 7. Figures

In [ ]:
MK = {'Latency-only': 'o-', 'Auth-only': 's--', 'Auth+Trust': '^-.', 'TAPS-MCC': 'D-'}
def plot(S, x, y, xl, yl, fn, pct=False):
    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    for s in STRATS:
        d = S[S.strategy == s]; k = 100 if pct else 1
        ax.errorbar(d[x], d[y + '_mean'] * k, yerr=d[y + '_ci95'] * k, fmt=MK[s], ms=4, capsize=2, label=s)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(alpha=.3); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f'sim/{fn}.png', dpi=300); plt.show()
plot(SA, 'users', 'failure_rate', 'Number of mobile users', 'Task failure rate (%)', 'fig_A_failure', True)
plot(SA, 'users', 'latency_ms', 'Number of mobile users', 'Service latency (ms)', 'fig_A_latency')
plot(SA, 'users', 'energy_J', 'Number of mobile users', 'Device energy per task (J)', 'fig_A_energy')
SB2 = SB.assign(mal_pct=SB.mal_ratio * 100)
plot(SB2, 'mal_pct', 'breach_rate', 'Malicious edge servers (%)', 'Security breach rate (%)', 'fig_B_breach', True)
plot(SB2, 'mal_pct', 'malicious_offload_rate', 'Malicious edge servers (%)', 'Malicious-offload rate (%)', 'fig_B_malicious_offload', True)
fig, ax = plt.subplots(figsize=(4.2, 3.2))
for (s, e), d in TR.groupby(['strategy', 'es']):
    ax.plot(d.time_s, d.trust, '-' if s == 'TAPS-MCC' else '--', lw=1.2, label=f'{s}: {e}')
ax.axhline(P['THETA'], color='k', lw=.8, ls=':', label='θ = 0.6')
ax.set_xlabel('Simulation time (s)'); ax.set_ylabel('Trust score TS'); ax.grid(alpha=.3); ax.legend(fontsize=6)
fig.tight_layout(); fig.savefig('sim/fig_C_trust.png', dpi=300); plt.show()

## 8. Save and download

In [ ]:
json.dump(dict(P=P, APPS=APPS, SEEDS=SEEDS), open('sim/parameters.json', 'w'), indent=2)
with zipfile.ZipFile('TAPS_SIM_results.zip', 'w') as z:
    for f in glob.glob('sim/*'): z.write(f)
try:
    from google.colab import files; files.download('TAPS_SIM_results.zip')
except Exception:
    print('Download TAPS_SIM_results.zip from the Files panel on the left.')